# Simulate UCJ circuits

In [1]:
import ffsim
import matplotlib.pyplot as plt
import numpy as np
import pyscf
import pyscf.cc
import pyscf.mcscf
import qiskit
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.primitives import StatevectorSampler
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler

In [2]:
atom: str = "H"
natoms: int = 6

In [3]:
def generate_linear_geometry(atom: str, natoms: int, atomic_distance: float = 1.0) -> str:
    """Returns a linear Hydrogen chain geometry for use in PySCF molecule construction.
    
    Args:
        natoms: Number of Hydrogen atoms in the chain.
        atomic_distance: Equal spacing between Hydrogen atoms.
    """
    return "; ".join([f"{atom} 0 0 {i * atomic_distance}" for i in range(natoms)])

In [4]:
# Specify molecule properties
spin_sq = 0

# Build N2 molecule
mol = pyscf.gto.Mole()
mol.build(
    atom=generate_linear_geometry(atom, natoms),
    basis="sto-6g",
)

# Define active space
n_frozen = 0
active_space = range(n_frozen, mol.nao_nr())

# Get molecular integrals
scf = pyscf.scf.RHF(mol).run()
norb = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
n_alpha = (n_electrons + mol.spin) // 2
n_beta = (n_electrons - mol.spin) // 2
nelec = (n_alpha, n_beta)
cas = pyscf.mcscf.CASCI(scf, norb, nelec)
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Compute exact energy using FCI
# reference_energy = cas.run().e_tot

print(f"norb = {norb}")
print(f"nelec = {nelec}")

converged SCF energy = -3.15600092954731
norb = 6
nelec = (3, 3)


In [5]:
# Get CCSD t2 amplitudes for initializing the ansatz
ccsd = pyscf.cc.CCSD(
    scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]
).run()
t1 = ccsd.t1
t2 = ccsd.t2

E(CCSD) = -3.257214530748514  E_corr = -0.101213601201206


In [6]:
import warnings

from qiskit.transpiler import CouplingMap

warnings.formatwarning = lambda msg, *args, **kwargs: f"Warning: {msg}\n"

# Set ansatz properties
n_reps = 1
pairs_aa = [(p, p + 1) for p in range(norb - 1)]
pairs_ab = None  # Let generate_lucj_pass_manager determine the alpha-beta interactions

# Initialize backend
coupling_map = CouplingMap.from_grid(
    num_rows=int(np.ceil(np.sqrt(2 * norb))),
    num_columns=int(np.ceil(np.sqrt(2 * norb)))
)
backend = GenericBackendV2(
    coupling_map.size(),
    coupling_map=coupling_map,
    basis_gates=["cp", "xx_plus_yy", "p", "x", "swap"],
)

# Create pass manager
try:
    pass_manager, pairs_ab = ffsim.qiskit.generate_lucj_pass_manager(
        backend=backend,
        norb=norb,
        connectivity="heavy-hex",
        interaction_pairs=(pairs_aa, pairs_ab),
        optimization_level=3,
    )
    print("Unable to generate ffsim pass manager")
except RuntimeError:
    pass_manager = None

# Create the LUCJ ansatz operator
ucj_op = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t2=t2,
    t1=t1,
    n_reps=n_reps,
    interaction_pairs=(pairs_aa, pairs_ab),
    # Setting optimize=True enables the "compressed" factorization
    optimize=True,
    # Limit the number of optimization iterations to prevent the code cell from running
    # too long. Removing this line may improve results.
    options=dict(maxiter=1000),
)

# create an empty quantum circuit
qubits = QuantumRegister(2 * norb, name="q")
circuit = QuantumCircuit(qubits)

# prepare Hartree-Fock state as the reference state and append it to the quantum circuit
circuit.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec), qubits)

# apply the UCJ operator to the reference state
circuit.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op), qubits)
# circuit.measure_all()

Unable to generate ffsim pass manager


In [7]:
if pass_manager:
    compiled = pass_manager.run(circuit)
else:
    compiled = qiskit.transpile(circuit, backend=backend)

In [8]:
print(f"Number of qubits: {compiled.num_qubits}")
print(f"Gate counts: {compiled.count_ops()}")

Number of qubits: 16
Gate counts: OrderedDict([('xx_plus_yy', 48), ('cp', 12), ('p', 12), ('x', 6), ('swap', 4)])


In [9]:
compiled.draw(fold=-1)

┌───────────────────────────┐                                                                                                                                                                                                                                                      ┌──────────────────────────┐                             ┌───────────────────────────┐                                                                                         ┌───────────────────────────┐        ┌────────────┐                                                                                                                                          
       q_5 -> 0 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤0                          ├─────────────────────────────────────■────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤0                         ├─────────────────────────────┤1                          ├─────────────────────────────────────────────────────────────────────────────────────────┤1                          ├────────┤ P(0.77729) ├──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                                                                                                             ┌──────────────────────────┐│  (XX+YY)(1.9411,-0.55608) │┌──────────────────────────┐         │P(0.65062)                                                                                                                                                                                                      │  (XX+YY)(2.4141,-3.7233) │ ┌─────────────────────────┐ │  (XX+YY)(0.60863,0.58722) │                                                            ┌──────────────────────────┐ │  (XX+YY)(2.4579,-0.62995) │        └────────────┘       ┌────────────────────────────┐        ┌───────────┐                                                                                
       q_4 -> 1 ─────────────────────────────────────────────────────────────────────────────────────────────┤0                         ├┤1                          ├┤0                         ├─────────■───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────■───────────────────────────────────────────────────────────────■────────────────────┤1                         ├─┤0                        ├─┤0                          ├────────────────────────────────────────────────────────────┤1                         ├─┤0                          ├─────────────────────────────┤1                           ├────────┤ P(2.0512) ├────────────────────────────────────────────────────────────────────────────────
                                                                                                             │                          │├───────────────────────────┤│                          │                                                                                                                                     │                                                               │P(-4.0721)          ├──────────────────────────┤ │                         │ └───────────────────────────┘                             ┌───────────────────────────┐  │                          │ └───────────────────────────┘┌───────────────────────────┐│                            │        ├───────────┴┐                                                                               
      q_11 -> 2 ─────────────────────────────────────────────────────────────────────────────────────────────┤                          ├┤0                          ├┤                          ├───────────────────────────────────────■──────────────────

## Compute observables with Qiskit

In [10]:
from qiskit.quantum_info import Statevector, SparsePauliOp

In [20]:
statevector = Statevector(compiled)
print(statevector)

observable = SparsePauliOp("ZZZZ")
sv_expectation_value = statevector.expectation_value(observable).real
print(sv_expectation_value)

Statevector([0.+0.j, 0.+0.j, 0.+0.j, ..., 0.+0.j, 0.+0.j, 0.+0.j],
            dims=(2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2))
0.9510861188147859


## Compute observables with Propaq

In [21]:
from propaq.datatypes._abstract import BitMask

from propaq.datatypes.majorana import MajoranaMonomial

from propaq.propagators import MajoranaPropagator
from propaq.circuits import MajoranaCircuit 
from propaq.noise import NoiselessModel, UniformNoiseModel, truncation
from propaq.noise import TruncationPolicy 

from propaq.datatypes import MajoranaTermSum

In [22]:
mc = MajoranaCircuit.from_qiskit(compiled, n_modes=4 * norb)

In [37]:
phase, majorana = MajoranaMonomial(7, 4 * norb) @ MajoranaMonomial(8, 4 * norb)
majorana

In [38]:
# TODO: Define observable from SparsePauliOp / PauliOp.
observable = MajoranaTermSum(
    {majorana: phase}
)

In [39]:
noise_model = UniformNoiseModel(0.00)  # NoiselessModel()

In [40]:
truncator = TruncationPolicy(weight_cutoff=np.inf, coeff_cutoff=1e-16)

In [41]:
prop = MajoranaPropagator(noise_model, truncator)

In [42]:
mp_expectation_value = prop.expectation_value(observable, mc, fock_state=0)
mp_expectation_value

Applying gate 1 / 162
Obs has 1 terms
Applying gate 2 / 162
Obs has 1 terms
Applying gate 3 / 162
Obs has 1 terms
Applying gate 4 / 162
Obs has 1 terms
Applying gate 5 / 162
Obs has 1 terms
Applying gate 6 / 162
Obs has 1 terms
Applying gate 7 / 162
Obs has 1 terms
Applying gate 8 / 162
Obs has 1 terms
Applying gate 9 / 162
Obs has 1 terms
Applying gate 10 / 162
Obs has 1 terms
Applying gate 11 / 162
Obs has 1 terms
Applying gate 12 / 162
Obs has 1 terms
Applying gate 13 / 162
Obs has 1 terms
Applying gate 14 / 162
Obs has 1 terms
Applying gate 15 / 162
Obs has 1 terms
Applying gate 16 / 162
Obs has 1 terms
Applying gate 17 / 162
Obs has 1 terms
Applying gate 18 / 162
Obs has 1 terms
Applying gate 19 / 162
Obs has 1 terms
Applying gate 20 / 162
Obs has 1 terms
Applying gate 21 / 162
Obs has 1 terms
Applying gate 22 / 162
Obs has 1 terms
Applying gate 23 / 162
Obs has 1 terms
Applying gate 24 / 162
Obs has 1 terms
Applying gate 25 / 162
Obs has 1 terms
Applying gate 26 / 162
Obs has 1 t

0.0